In [10]:
import numpy as np
from numpy.linalg import norm
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
import os
from pypdf import PdfReader
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from pathlib import Path

dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

True

In [11]:
HF_KEY = os.getenv("HF_TOKEN")
GITHUB_KEY = os.getenv("GITHUB_TOKEN")

In [4]:
# ── 1. READ PDF ───────────────────────────────────────────────────────────
def read_pdf(path: str) -> tuple[str, int]:
    reader = PdfReader(path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n\n"  # double newline between pages
    return full_text, len(reader.pages)

In [5]:
# ── 2. CLEAN ──────────────────────────────────────────────────────────────
def clean(text: str) -> str:
    text = re.sub(r'#{1,6}\s*', '', text)               # remove ## headings
    text = re.sub(r'\*{1,2}(.*?)\*{1,2}', r'\1', text)  # remove **bold**
    text = re.sub(r'\n{3,}', '\n\n', text)               # collapse excess newlines
    text = re.sub(r'[ \t]+', ' ', text)                  # collapse spaces
    return text.strip()

In [14]:
import time
hf_client = InferenceClient(provider="hf-inference", api_key=HF_KEY)

def embed_texts(texts: list[str], model_name) -> list[list[float]]:
    embeddings = []
    for text in texts:
        vector = np.array(hf_client.feature_extraction(
            text,
            model=model_name,
        ))
        # Fix shape if (1, 768) instead of (768,)
        if vector.ndim > 1:
            vector = vector.squeeze()
        vector = vector / norm(vector)
        embeddings.append(vector.tolist())
        time.sleep(0.1)
    return embeddings

from azure.ai.inference import EmbeddingsClient
from azure.core.credentials import AzureKeyCredential

openai_embeddings_client = EmbeddingsClient(
    endpoint="https://models.github.ai/inference",
    credential=AzureKeyCredential(GITHUB_KEY)
)

def embed_openai_large(texts: list[str]) -> list[list[float]]:
    embeddings = []
    for text in texts:
        response = openai_embeddings_client.embed(
            input=[text],
            model="openai/text-embedding-3-large"
        )
        vector = np.array(response.data[0].embedding)
        # Normalize (OpenAI recommends normalizing for cosine similarity)
        vector = vector / np.linalg.norm(vector)
        embeddings.append(vector.tolist())
        time.sleep(0.1)
    return embeddings


In [15]:
# ── 3. CHUNK with LangChain ───────────────────────────────────────────────
def split_into_chunks(text: str) -> list[str]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=250,
        length_function=len, # بنعرف المودل على اسم الفانكشن الي بتطلع الطول
        separators=["\n\n", "\n", ".", "،", " ", ""]
        )
    chunks = text_splitter.split_text(text)
    return chunks

In [16]:
# Step 1 — extract real chunks from your PDF
raw_text, _ = read_pdf("Prototype_UCAS_Documents.pdf")
cleaned     = clean(raw_text)
real_chunks = split_into_chunks(cleaned)

print(f"Total real chunks: {len(real_chunks)}")
print(f"\nExample chunk:\n{real_chunks[0][:300]}")

Total real chunks: 4

Example chunk:
نظرة عامة 
البرنامج يؤهل الملتحقين به للتعامل مع البيانات الكبيرة وتحليلها إحصائيا، واستنباط المعرفة منها، مما يساعد في 
اتخاذ القرارات بطريقة دقيقة وعلمية. 
ويذكر أن علم البيانات والذكاء الاصطناعي يُعدُّ التخصص الأكثر تطوراً في العالم، خلال السنوات الأخيرة، وأن 
معظم الدول المتقدمة، وفي ظل الثورة ا


In [17]:
for chunck in real_chunks:
    print(f"Chunck: {chunck[:300]}")

Chunck: نظرة عامة 
البرنامج يؤهل الملتحقين به للتعامل مع البيانات الكبيرة وتحليلها إحصائيا، واستنباط المعرفة منها، مما يساعد في 
اتخاذ القرارات بطريقة دقيقة وعلمية. 
ويذكر أن علم البيانات والذكاء الاصطناعي يُعدُّ التخصص الأكثر تطوراً في العالم، خلال السنوات الأخيرة، وأن 
معظم الدول المتقدمة، وفي ظل الثورة ا
Chunck: الشرق الأوسط كالأردن والامارات والسعودية والبحرين. إضافة الى ان استخدام علم البينات والذكاء الاصطناعي 
في ارتفاع مملوس في كل من الأعمال التجارية خصوصا والعالم بشكل عام. 
 
فرص العمل 
يستطيع مختص علم البيانات والذكاء الاصطناعي أن يمارس عمله منفرداً أو ضمن فريق عمل سواء في قطاع 
خاص أو قطاع عام، وغالب
Chunck: زوارنا الكرام .. طلبتنا الأعزاء 
نستعرض معكم حزمة المنح الدراسية والتسهيلات المقدمة للطلبة الجدد للعام الجامعي 2022-2023 م 
 
 
منحة طلبة الثانوية العامة المتفوقين 
• منحة دراسية 100% لكافة الاختصاصات 
للطلبة الحاصلين علي معدل الثانوية 95% فأكثر 
تستمر المنحة طيلة الدراسة بمعدل فصلي 95 % 
• منحة درا
Chunck: منحة75% لمستفيدي الشؤون الاجتماعية للعديد من الاختصاصات 
• تستمر 

In [18]:
test_pairs = [
    (real_chunks[0],   "query: هل يمكنك إخباري بشكل أكبر عن تخصص علم البيانات والذكاء الاصطناعي؟",       1),
    (real_chunks[1], "query: ما هي متطلبات القبول في البرنامج؟",       0),
    (real_chunks[2],   "query: هل يوجد منح دراسية للطلاب المتفوقين؟",    1),
    (real_chunks[0], "query: ما هو موعد امتحانات نهاية الفصل؟",        0),
]

In [19]:
def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (norm(v1) * norm(v2)))

results = []

for chunk, query, label in test_pairs:
    # Model 1
    sim_matryoshka = cosine_similarity(
        embed_texts([chunk], "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2")[0],
        embed_texts([query], "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2")[0]
    )
    # Model 2
    sim_e5 = cosine_similarity(
        embed_texts([f"passage: {chunk}"], "intfloat/multilingual-e5-large")[0],
        embed_texts([f"query: {query}"], "intfloat/multilingual-e5-large")[0]
    )
    # Model 3
    sim_openai = cosine_similarity(
        embed_openai_large([chunk])[0],   # OpenAI doesn't use passage/query prefixes
        embed_openai_large([query])[0]
    )
    # Model 4
    sim_bge_m3 = cosine_similarity(
        embed_texts([chunk], "BAAI/bge-m3")[0],
        embed_texts([query], "BAAI/bge-m3")[0]
    )

    results.append({
        "query":         query,
        "label":         label,
        "matryoshka":    round(sim_matryoshka, 4),
        "e5_large":      round(sim_e5, 4),
    })
    print(f"{'✅' if label == 1 else '❌'} {query[:50]}")
    print(f"   Matryoshka : {sim_matryoshka:.4f}\n")
    print(f"   E5-large   : {sim_e5:.4f}\n")
    print(f"   OpenAI-3L  : {sim_openai:.4f}\n")
    print(f"   BGE-M3     : {sim_bge_m3:.4f}\n")

✅ query: هل يمكنك إخباري بشكل أكبر عن تخصص علم البيا
   Matryoshka : 0.6727

   E5-large   : 0.8790

   OpenAI-3L  : 0.5927

   BGE-M3     : 0.6547

❌ query: ما هي متطلبات القبول في البرنامج؟
   Matryoshka : 0.0778

   E5-large   : 0.7800

   OpenAI-3L  : 0.2293

   BGE-M3     : 0.3870

✅ query: هل يوجد منح دراسية للطلاب المتفوقين؟
   Matryoshka : 0.5872

   E5-large   : 0.8364

   OpenAI-3L  : 0.5409

   BGE-M3     : 0.6824

❌ query: ما هو موعد امتحانات نهاية الفصل؟
   Matryoshka : -0.1104

   E5-large   : 0.7353

   OpenAI-3L  : 0.1508

   BGE-M3     : 0.3602

